# Weather Hazard Category · MLP Classifier

**36120 Advanced Machine Learning Application** · Assessment Task 2 · Weather Intelligence (Open-Meteo, Sydney)

*Experiment notebook · predicts WHC exactly +7 days ahead*

This notebook trains and evaluates the **MLP Classifier** model for the **Weather Hazard Category (WHC)**, using the leakage-audited feature matrix produced by the preprocessing notebook.

### Student details

| | |
|:--|:--|
| **Student name** | Juan Vargas T. |
| **Student ID** | 25945320 |
| **Subject** | 36120 Advanced Machine Learning Application |
| **Assessment** | AT2 · Weather Intelligence |
| **GitHub** | [Sebas102507](https://github.com/Sebas102507) |

## Overview

### Hypothesis

A small multi-layer perceptron will **not beat the tree ensembles or the linear model** on macro
F1: the minority classes offer too few examples for a network to learn a boundary around them,
and `MLPClassifier` has no class-weighting option to compensate.

That limitation is the reason to run the experiment: it isolates how much of any ensemble's
advantage came from class weighting rather than from the model family.

### Design

| Choice | Value | Reason |
|:--|:--|:--|
| Algorithm | `StandardScaler` → `MLPClassifier` | Smooth decision boundaries; scaling stays inside the fit |
| Imbalance | **No mechanism available** | `MLPClassifier` has no `class_weight`; a stated limitation, not an oversight |
| Early stopping | 15% of the *training* data, 20 rounds patience | The project's validation split is never used for fitting |
| Strategy | One classifier for the single +7 day horizon | Matches the deployed artefact layout |
| Selection | Optuna TPE sampler, 20 trials per target | See section 6 |
| Objective | validation **macro F1** (higher is better) | See section 6 |

### Workflow

All ten experiment notebooks follow the same steps and run the same code from
`nucleus_ml.weather.Experiment`, so their results are directly comparable. **Only the model
specification in section 1 differs between notebooks.**

| # | Step | Purpose |
|:-:|:--|:--|
| 1 | Model specification | The only algorithm-specific cell |
| 2 | Data | Training and validation splits (the test split is never loaded here) |
| 3 | Baselines | The floor every model must clear |
| 4 | Default configuration | An untuned reference point |
| 5 | Feature-family ablation | What each block of engineered features contributes |
| 6 | Hyperparameter selection | Choose settings on the validation split |
| 7 | Seed stability | Is the tuning gain real or noise? |
| 8 | Final model | Tuned model against the default and the baselines |
| 9 | Feature importance | Which features the model relies on |
| 10 | Generalisation check | Training vs validation performance |
| 11 | Save artefacts | Model bundle + `/model-metadata` document |
| 12 | Conclusions | What to carry into the report |

---
## 0. Setup

Imports, the experiment's identity and the task definition. Everything else comes from the
project package, so no helper is redefined in this notebook.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from nucleus_ml import weather as wx

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", "{:.3f}".format)

TASK = "weather_hazard"
EXPERIMENT = "whc_mlp"
ALGORITHM = "MLP Classifier"
N_TRIALS = 20  # Optuna trials per target (None when a fixed grid is used)

task = wx.load_task(TASK)
print(f"{task.target_name} | {ALGORITHM} | {len(task.feature_columns)} features | "
      f"targets: {task.target_columns}")

---
## 1. Model specification

This is the **only algorithm-specific cell**. It defines how the estimator is built, a sensible
default configuration, a cheaper configuration for the ablation, and the candidate settings for
section 6 (either an Optuna search space or a fixed grid).

Oversampling the rare days is the usual workaround for missing class weights, but duplicating a
few dozen days across 2,900 rows invites memorisation, so it is not applied. `alpha` does double
duty here: heavy regularisation also pushes the network toward the majority class.

In [ ]:
def build_estimator(params: dict, seed: int) -> Pipeline:
    """Scale the features, then fit a multi-layer perceptron classifier."""
    params = dict(params)
    layers = (params.pop("units"),) * params.pop("n_layers")
    return Pipeline([
        ("scale", StandardScaler()),
        ("mlp", MLPClassifier(hidden_layer_sizes=layers, max_iter=800, early_stopping=True,
                              n_iter_no_change=20, validation_fraction=0.15,
                              random_state=seed, **params)),
    ])


DEFAULT_PARAMS = {"n_layers": 2, "units": 128, "alpha": 1e-3, "learning_rate_init": 1e-3,
                  "activation": "relu"}
ABLATION_PARAMS = {**DEFAULT_PARAMS, "n_layers": 1, "units": 32}


def search_space(trial) -> dict:
    """Network size, L2 penalty, learning rate and activation."""
    return {
        "n_layers": trial.suggest_int("n_layers", 1, 3),
        "units": trial.suggest_int("units", 16, 256, log=True),
        "alpha": trial.suggest_float("alpha", 1e-6, 1e-1, log=True),
        "learning_rate_init": trial.suggest_float("learning_rate_init", 1e-4, 2e-2, log=True),
        "activation": trial.suggest_categorical("activation", ["relu", "tanh"]),
    }


SEARCH_SPACE, GRID = search_space, None

In [ ]:
exp = wx.Experiment(
    task,
    name=EXPERIMENT,
    algorithm=ALGORITHM,
    build=build_estimator,
    default_params=DEFAULT_PARAMS,
    ablation_params=ABLATION_PARAMS,
    search_space=SEARCH_SPACE,
    grid=GRID,
)

---
## 2. Data

Only the **training** and **validation** splits are loaded. The test split is reserved for a
single evaluation in the comparison notebook, once one model has been selected, so nothing in
this notebook can be influenced by it.

> The preprocessing notebook already applied the chronological split, the warm-up filter and an
> embargo at each boundary. Nothing is re-split or shuffled here.

In [ ]:
exp.data_summary()

---
## 3. Baselines

Three naive forecasts, all fitted on the training split only:

| Baseline | Rule | What it knows |
|:--|:--|:--|
| **Majority class** | Always predict the most common class | Nothing but the class balance |
| **Persistence** | The class in 7 days equals today's class (from today's WHI) | The current weather |
| **Seasonal mode** | The most common class for that month | The season |

The majority-class rule scores a high *accuracy* but a low *macro F1*, which is exactly why
accuracy is not used to select models for this imbalanced target.

In [ ]:
exp.run_baselines()

---
## 4. Default configuration

The untuned default is scored before any search. It is the reference for judging what tuning
bought in section 8; a search that fails to beat a sensible default is a result worth reporting.
`best_baseline` is repeated for comparison.

In [ ]:
exp.run_defaults()

---
## 5. Feature-family ablation

Feature families are added **one at a time**, starting from persistence alone (a single column),
and the model is refitted at each step. This measures what each block of engineering contributes
instead of assuming it. The cheaper `ABLATION_PARAMS` are used throughout so the differences come
from the features, not from tuning. Scores are on validation (macro F1).

In [ ]:
ablation = exp.run_ablation()
plt.show()
ablation

---
## 6. Hyperparameter selection

The objective is validation **macro F1** (higher is better).

**Why macro F1.** It averages F1 across classes without weighting by frequency, so a model that
ignores the rare hazard classes cannot hide behind the common Low Risk days. Accuracy would reward
exactly that behaviour. The average is taken over the classes present in training, because a
class the model never saw cannot be predicted.

**Why validation, not cross-validation.** The split is chronological. Shuffled k-fold would put
future days into training and inflate every score, because consecutive days are strongly
correlated. The fixed 2024 validation year matches how the model will be used: trained on the
past, applied forward.

> **Stated plainly:** selecting on validation makes the validation score slightly optimistic. The
> honest estimate comes from the test split, which only the comparison notebook reads, once.

In [ ]:
exp.tune(n_trials=N_TRIALS)

In [ ]:
exp.plot_search()
plt.show()

---
## 7. Seed stability

The selected configuration is refitted with three different random seeds. If the score moves by
as much as the spread between the top search candidates (`search_beats_noise = False`), the search
was ranking noise and the conclusion should say so. Deterministic models show a spread of zero.

In [ ]:
exp.seed_stability()

---
## 8. Final model against the baselines

The selected configuration is refitted and compared with the default configuration and every
baseline. `gain_of_tuned_model` is positive when the tuned model is better than that row.

Read the **confusion matrix** as well as the headline score: a model that almost never predicts
the rare classes has not learned to flag hazardous days, whatever its macro F1. The printed
boundary summary shows how many errors sit within a few WHI points of a class threshold; those are
threshold effects rather than genuine misses.

In [ ]:
exp.finalise()

In [ ]:
exp.plot_validation()
plt.show()

---
## 9. Feature importance

This model has no built-in importance, so **permutation importance** is used: each feature is shuffled on the validation split and the drop in score is recorded. Features with negative importance are candidates for pruning.

> Correlated features (a variable, its lags and its rolling means) share importance, so a feature
> that looks unimportant on its own may belong to an essential block. The ablation in section 5 is
> the better guide to what can be removed.

In [ ]:
family_importance = exp.importance()
plt.show()
family_importance

---
## 10. Generalisation check

Training and validation scores side by side. A small gap means the model generalises; a large
gap means it has memorised patterns in the training years that do not carry over to 2024. Read
it together with the seed-stability table in section 7.

In [ ]:
exp.generalisation()

---
## 11. Save artefacts

The fitted model(s) are written to `models/weather_hazard/experiments/whc_mlp/` with:

| File | Contents |
|:--|:--|
| `whc_mlp_h<horizon>.joblib` | One fitted model per horizon |
| `metadata.json` | Target, algorithm, horizon, features, importance, hyperparameters, metrics — the `/model-metadata` document |
| `manifest.json` | File list and the feature column order the API must use |

> The feature column **order** is recorded because models receive an unnamed array at prediction
> time and would silently give wrong answers if the columns were permuted.

In [ ]:
exp.export()

---
## 12. Conclusions

- **The class-weighting comparison.** If the unweighted MLP matches the weighted ensembles,
  weighting was not what mattered; if it collapses onto the majority class, weighting was
  essential — a concrete recommendation for any future model.
- **Predicted class distribution.** Check the per-class table first: a network that predicts one
  class every day has a low macro F1 regardless of the search.
- **Cost against benefit.** The MLP is the least interpretable and most expensive option; it needs
  a clear macro F1 advantage over the logistic regression to justify deployment.

> The test split has still not been used. It is scored exactly once, in the
> `whc_comparison` notebook, after one model has been selected on validation
> performance across all five WHC experiments.